# Laboratory Task 4 — PyTorch Regression
Name: Emmanuel R. Sevilla  
DS4A

**Instruction:** Train a regression model in PyTorch with **MSE loss**, **2 fully connected layers**, **batch size 8**, the **SGD** optimizer, and **1000 epochs**.

The data is the same temperature / rainfall / humidity → apples / oranges dataset from the source notebook.

In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)

# inputs: temperature, rainfall, humidity
inputs = np.array([[73, 67, 43], [91, 88, 64], [87, 134, 58],
                   [102, 43, 37], [69, 96, 70]] * 3, dtype="float32")

# targets: apples, oranges
targets = np.array([[56, 70], [81, 101], [119, 133],
                    [22, 37], [103, 119]] * 3, dtype="float32")

## 1. Prepare the data

Both inputs and targets are standardized so SGD stays stable over 1000 epochs. The predictions are converted back to the original scale at the end.

In [2]:
x_mean, x_std = inputs.mean(axis=0), inputs.std(axis=0)
y_mean, y_std = targets.mean(axis=0), targets.std(axis=0)

X = torch.tensor((inputs - x_mean) / x_std)
Y = torch.tensor((targets - y_mean) / y_std)

dataset = TensorDataset(X, Y)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

print("Samples   :", len(dataset))
print("Batch size:", loader.batch_size)

Samples   : 15
Batch size: 8


## 2. Model with two fully connected layers

The model has two `nn.Linear` layers. No activation is placed between them because this is a linear regression, so the whole model stays linear.

In [3]:
model = nn.Sequential(
    nn.Linear(3, 8),
    nn.Linear(8, 2)
)
print(model)

Sequential(
  (0): Linear(in_features=3, out_features=8, bias=True)
  (1): Linear(in_features=8, out_features=2, bias=True)
)


## 3. Loss function and optimizer

In [4]:
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

## 4. Train for 1000 epochs

In [5]:
epochs = 1000

for epoch in range(1, epochs + 1):
    total_loss = 0
    for xb, yb in loader:
        loss = criterion(model(xb), yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(xb)

    if epoch % 100 == 0:
        print(f"Epoch {epoch:4d}/{epochs}  Loss: {total_loss / len(dataset):.6f}")

Epoch  100/1000  Loss: 0.002257
Epoch  200/1000  Loss: 0.000599
Epoch  300/1000  Loss: 0.000471
Epoch  400/1000  Loss: 0.000435
Epoch  500/1000  Loss: 0.000421
Epoch  600/1000  Loss: 0.000413
Epoch  700/1000  Loss: 0.000412
Epoch  800/1000  Loss: 0.000411
Epoch  900/1000  Loss: 0.000413
Epoch 1000/1000  Loss: 0.000411


## 5. Evaluate the model

In [6]:
with torch.no_grad():
    preds = model(X).numpy() * y_std + y_mean

mse = np.mean((preds - targets) ** 2)
print("MSE (original scale):", round(float(mse), 4))
print()
for i in range(5):
    print(f"Sample {i + 1}: predicted {np.round(preds[i], 2)}  actual {targets[i]}")

MSE (original scale): 0.4833

Sample 1: predicted [56.9  69.93]  actual [56. 70.]
Sample 2: predicted [ 82.36 100.92]  actual [ 81. 101.]
Sample 3: predicted [118.76 133.  ]  actual [119. 133.]
Sample 4: predicted [21.11 37.06]  actual [22. 37.]
Sample 5: predicted [101.86 119.09]  actual [103. 119.]


## Conclusion

The model follows every requirement: MSE loss, two fully connected layers, batch size 8, SGD, and 1000 epochs. The last cell shows the final MSE and compares the predictions with the real values.